# Garbage YOLO — baseline v0 (YOLO11n) w Google Colab
Detekcja i klasyfikacja śmieci (6 klas: BIODEGRADABLE, CARDBOARD, GLASS, METAL, PAPER, PLASTIC).

**Przed startem:** `Środowisko wykonawcze → Zmień typ środowiska → GPU (T4)`.
Notebook: klonuje repo → naprawia `data.yaml` (w repo ma ścieżkę z Maca) → trenuje YOLO11n → test → pobiera `best.pt`.
Hiperparametry identyczne jak w `train_baseline.ipynb` z repo, żeby baseline był porównywalny.

In [ ]:
!nvidia-smi -L
%pip install -q "ultralytics>=8.3,<9" "codecarbon>=3,<4" "pandas>=2,<4"

GPU 0: Tesla T4 (UUID: GPU-39f11928-ae28-a1a9-4036-59b9d2ad925c)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.3 MB/s eta 0:00:00


In [ ]:
!git clone --depth 1 https://github.com/PovMiki/garbage-yolo /content/garbage-yolo

Cloning into '/content/garbage-yolo'...
remote: Enumerating objects: 14594, done.
remote: Counting objects: 100% (14594/14594), done.
remote: Compressing objects: 100% (8096/8096), done.
remote: Total 14594 (delta 2441), reused 14590 (delta 2441), pack-reused 0 (from 0)
Receiving objects: 100% (14594/14594), 163.12 MiB | 18.16 MiB/s, done.
Resolving deltas: 100% (2441/2441), done.
Updating files: 100% (20258/20258), done.


In [ ]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
import json, uuid, yaml, shutil
import pandas as pd
import torch
from ultralytics import YOLO
from codecarbon import OfflineEmissionsTracker

# ---- KONFIGURACJA (baseline: nie zmieniaj, jeśli chcesz porównywać kolejne iteracje) ----
REPO = Path("/content/garbage-yolo")
DS = REPO / "dataset_clean"
EPOCHS, BATCH, IMGSZ, SEED = 50, 8, 640, 42
PATIENCE, LR0, OPTIMIZER = 10, 0.001, "AdamW"
COUNTRY = "POL"        # serwer Colab bywa w innym kraju - zmień, jeśli chcesz dokładniejszą estymację CO2
VERSION_TAG = "baseline_v0"
# ------------------------------------------------------------------------------------------

DEVICE = 0 if torch.cuda.is_available() else "cpu"
assert DEVICE == 0, "Brak GPU! Włącz GPU w ustawieniach środowiska wykonawczego."

# data.yaml z repo ma absolutną ścieżkę z Maca -> tworzymy poprawny plik dla Colaba
cfg = yaml.safe_load((DS / "data.yaml").read_text())
cfg["path"] = str(DS)
DATA = DS / "data_colab.yaml"
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print(cfg)
for s in ("train", "valid", "test"):
    print(s, len(list((DS / s / "images").glob("*"))), "zdjęć")

RUN_NAME = f"{VERSION_TAG}_" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
ROOT = Path("/content/runs"); RUN = ROOT / RUN_NAME
RUN.mkdir(parents=True)
(RUN / "environment.json").write_text(json.dumps({
    "packages": {p: version(p) for p in ("ultralytics", "torch", "codecarbon")},
    "device": torch.cuda.get_device_name(0), "country": COUNTRY, "seed": SEED,
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "optimizer": OPTIMIZER, "lr0": LR0, "patience": PATIENCE,
}, indent=2))
print("GPU:", torch.cuda.get_device_name(0), "| Wyniki:", RUN)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{'path': '/content/garbage-yolo/dataset_clean', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 6, 'names': {0: 'BIODEGRADABLE', 1: 'CARDBOARD', 2: 'GLASS', 3: 'METAL', 4: 'PAPER', 5: 'PLASTIC'}}
train 3533 zdjęć
valid 1008 zdjęć
test 505 zdjęć
GPU: Tesla T4 | Wyniki: /content/runs/baseline_v0_20261012_164706_9034cc


## Trening (z pomiarem CodeCarbon)

In [ ]:
model = YOLO("yolo11n.pt")
tracker = OfflineEmissionsTracker(
    project_name=RUN_NAME, country_iso_code=COUNTRY,
    output_dir=str(RUN), output_file="wyniki/emissions.csv",
    measure_power_secs=15, tracking_mode="machine",
    save_to_file=True, save_to_api=False, log_level="warning",
)
tracker.start()
try:
    model.train(
        data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=2, seed=SEED, deterministic=True,
        optimizer=OPTIMIZER, lr0=LR0, cos_lr=True, patience=PATIENCE,
        amp=True, cache=False,
        project=str(ROOT), name=RUN.name, exist_ok=True,
        save=True, plots=True, val=True,
    )
finally:
    co2_kg = tracker.stop()
    print("Szacowane CO2eq [kg]:", co2_kg)

[codecarbon INFO @ 16:47:07] offline tracker init
[codecarbon WARNING @ 16:47:07] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 16:47:09] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 16:47:09] We will use the default power consumption of 4 W per thread for your 2 CPU, so 8W.
[codecarbon WARNING @ 16:47:09] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 16:47:09] No CPU tracking mode found. Falling back on CPU load mode.


Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/garbage-yolo/dataset_clean/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_v0_20261012_164706_9034cc, nbs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Ocena na zbiorze test

In [ ]:
best = RUN / "weights/best.pt"
assert best.is_file(), "Brak best.pt"
metrics = YOLO(str(best)).val(
    data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, workers=2, project=str(RUN), name="test", plots=True,
)
scores = {"mAP50": float(metrics.box.map50), "mAP50-95": float(metrics.box.map),
          "precision": float(metrics.box.mp), "recall": float(metrics.box.mr)}
scores["per_class_mAP50-95"] = {metrics.names[int(c)]: float(metrics.box.maps[int(c)]) for c in metrics.box.ap_class_index}
(RUN / "test_metrics.json").write_text(json.dumps(scores, indent=2))
print(json.dumps(scores, indent=2))
display(pd.read_csv(RUN / "results.csv").tail())
if (RUN / "emissions.csv").is_file():
    display(pd.read_csv(RUN / "emissions.csv")[["emissions", "energy_consumed", "duration"]])

## Podgląd wyników

In [ ]:
import glob
from IPython.display import Image, display
for p in [str(RUN / "results.png"), str(RUN / "test/confusion_matrix_normalized.png")] + sorted(glob.glob(str(RUN / "val_batch*_pred.jpg")))[:1]:
    if Path(p).is_file(): display(Image(p, width=900))

## Pobranie plików
Osobno `best.pt` + zip z metrykami, wykresami i raportem emisji.

In [ ]:
from google.colab import files
out = Path("/content/export"); out.mkdir(exist_ok=True)
shutil.copy(best, out / f"{VERSION_TAG}_best.pt")
shutil.copy(RUN / "weights/last.pt", out / f"{VERSION_TAG}_last.pt")
for f in ("results.csv", "results.png", "args.yaml", "environment.json", "test_metrics.json", "emissions.csv"):
    if (RUN / f).is_file(): shutil.copy(RUN / f, out / f)
if (RUN / "test/confusion_matrix_normalized.png").is_file():
    shutil.copy(RUN / "test/confusion_matrix_normalized.png", out / "test_confusion_matrix_normalized.png")
zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", out)
files.download(str(out / f"{VERSION_TAG}_best.pt"))
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>